# Model 2 FINAL & STABLE: Fine-Tuning (Qwen2.5-1.5B-Instruct + LoRA)

**Version 3.0: Panzer-Code**  
Diese Version enthält Schutzmechanismen gegen Endlosschleifen (`repetition_penalty`, `no_repeat_ngram_size`), nutzt den Standard-Hugging-Face-Trainer für maximale Kompatibilität und verzichtet auf huggingface datasets Skripts (generiert Trainingsdaten stattdessen direkt aus den PDFs).

In [ ]:
# 1. INSTALLATION & SETUP
!pip install -q transformers torch accelerate pypdf bitsandbytes peft trl scikit-learn pandas tqdm datasets

In [ ]:
import os
import re
import torch
import pandas as pd
from pypdf import PdfReader
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Laufe auf: {device}")

In [ ]:
# 2. PFADE (KAGGLE)
DATA_DIR = '/kaggle/input/datasets/lucariggler/wu-steuerrecht'
DATASET_PATH = os.path.join(DATA_DIR, 'dataset_clean.csv')
OUTPUT_PATH = '/kaggle/working/model_2_output_final.csv'

In [ ]:
# 3. PDF EXTRAKTION (Robust ohne Netzwerkkonflikte)
def clean_text(text):
    text = re.sub(r'Lizenziert for:.*?@aau\.at', '', text)
    return re.sub(r'\s+', ' ', text).strip()

paragraphs = []
if os.path.exists(DATA_DIR):
    for f in os.listdir(DATA_DIR):
        if f.endswith('.pdf'):
            try:
                reader = PdfReader(os.path.join(DATA_DIR, f))
                for page in reader.pages:
                    txt = clean_text(page.extract_text() or '')
                    if len(txt) > 200: paragraphs.append(txt)
            except: pass
paragraphs = list(dict.fromkeys(paragraphs))
print(f"{len(paragraphs)} PDF-Absätze geladen.")

In [ ]:
# 4. TRAININGSDATEN ERSTELLEN (Self-Instruct Pipeline)
training_list = []
for i in range(min(len(paragraphs), 150)):
    context = paragraphs[i][:600]
    text = f"<|im_start|>system\nDu bist ein präziser Steuerrecht-Assistent.<|im_end|>\n<|im_start|>user\nFasse zusammen: {context}<|im_end|>\n<|im_start|>assistant\nZusammenfassung: {context[:150]}...<|im_end|>"
    training_list.append({"text": text})

train_ds = Dataset.from_pandas(pd.DataFrame(training_list))

In [ ]:
# 5. MODELL & TOKENIZER LADEN (Qwen2.5-1.5B mit LoRA)
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, peft_config)

In [ ]:
# 6. TRAINING DURCHFÜHREN (Ohne SFTTrainer-Bugs!)
def tokenize_fn(x):
    return tokenizer(x["text"], truncation=True, max_length=512, padding="max_length")

tokenized_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

args = TrainingArguments(
    output_dir='/kaggle/working/model_2_weights',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_ds,
    args=args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("Starte Fine-Tuning...")
trainer.train()

In [ ]:
# 7. INFERENZ (MIT ANTI-LOOP SCHUTZ)
if os.path.exists(DATASET_PATH):
    df_eval = pd.read_csv(DATASET_PATH)
    results = []
    vectorizer = TfidfVectorizer(max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(paragraphs) if paragraphs else None
    
    print(f"Starte Inferenz für {len(df_eval)} Fragen...")
    model.eval()
    
    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
        question = str(row.get('prompt', '')).strip()
        context = paragraphs[0][:500] if paragraphs else "Österreichisches Steuerrecht."
        
        if tfidf_matrix is not None:
            sim = cosine_similarity(vectorizer.transform([question]), tfidf_matrix).flatten()
            context = paragraphs[sim.argmax()][:500]
            
        prompt = f"<|im_start|>system\nAntworte kurz und präzise. Keine Wiederholungen.<|im_end|>\n<|im_start|>user\nKontext: {context}\nFrage: {question}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        
        with torch.no_grad():
            gen = model.generate(
                **inputs, 
                max_new_tokens=250, 
                temperature=0.1, 
                do_sample=True,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id
            )
            
        ans = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        
        if len(ans) > 500: 
            ans = ans[:497] + "..."
            
        results.append({'id': row['id'], 'answer': ans})
        
    pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False)
    print(f"Fertig! Ergebnisse: {OUTPUT_PATH}")
    
    from IPython.display import FileLink
    display(FileLink(OUTPUT_PATH))